[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adaptionlabs/adaption-runbooks/blob/main/adaptive-data/adaptive_data_multimodal.ipynb)

# Beyond Text: Adaptive Data for the Multimodal Datasets

Adapt a vision dataset with Adaptive Data and prepare it for training, using the same API and Python SDK you already use for text and documents.

**Docs:** [Configure Multimodal Datasets](https://docs.adaptionlabs.ai/adaptive-data/select-columns/#map-an-image-column)

**Blog:** [Beyond Text: Adaptive Data for the Multimodal Era](https://adaptionlabs.ai/blog/adaptive-data-multimodal)

## Overview

A vision model is only ever as current as the data behind it, and the images it sees in production are rarely the images it was trained on. A hospital swaps in new scanners and a diagnostic model starts missing findings it used to catch; a client sends contracts in a new layout and the review model falls apart. Real progress happens in the long tail, where the rare, the nuanced, and the domain-specific live. That is exactly where static datasets fall short, and what Adaptive Data is built to reach.

Adaptive Data for vision supports the image tasks teams depend on most: **visual question answering**, **image captioning**, **visual reasoning**, **image classification**, and **document question answering**. Bring images as URLs, embedded bytes, or file references, together with the text that goes with each one (a question and answer, a caption, a label), and get back enhanced versions of the same dataset in the format you already consume. The capabilities you already rely on carry over to images:

- **Translation and localization:** grow the text paired with your images across 242 languages and localizations.
- **Blueprint:** set tone, length, safety thresholds, and custom content policies, and every example you get back is shaped and enforced against them.

Across five vision datasets spanning charts, finance, captioning, numerical, and spatial QA, adapted data has an average **67% win rate** over the baseline. It is the same approach that has delivered an average 82% increase in data quality in text and documents.

This runbook adapts [AyaVisionBench](https://huggingface.co/datasets/CohereLabs/AyaVisionBench), a multilingual vision-language benchmark, end to end: **import** the dataset, **map** its prompt, reference answer, and image columns, set a **blueprint**, **estimate** the cost, **adapt**, **check quality**, and **download** the result.

In [1]:
%pip install -q adaption pandas pyarrow

/Users/andrijazz/source/adaption-runbooks/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## Setup

Load your API key. In Colab, add `ADAPTION_API_KEY` to the secrets panel (in the sidebar) and enable it for this notebook. Running locally, the cell prompts for the key if the environment variable is not set.

In [2]:
import os

from adaption import Adaption

if "ADAPTION_API_KEY" not in os.environ:
    try:
        from google.colab import userdata  # Colab secrets

        os.environ["ADAPTION_API_KEY"] = userdata.get("ADAPTION_API_KEY")
    except ImportError:
        from getpass import getpass

        os.environ["ADAPTION_API_KEY"] = getpass("ADAPTION_API_KEY: ")

client = Adaption()

## Parameters

Everything you might change lives here, so the rest of the notebook runs unedited.

In [3]:
# Source dataset to import from Hugging Face (AyaVisionBench is a text-image dataset).
DATASET_URL = "https://huggingface.co/datasets/CohereLabs/AyaVisionBench"
DATASET_FILES = ["eng_Latn/test-00000-of-00001.parquet"]

# Map source columns to the roles Adaptive Data expects.
COLUMN_MAPPING = {
    "prompt": "prompt",
    "completion": "reference",
    "image": "image",
}

# Keep runs cheap while you iterate. Set to None to process the whole dataset.
MAX_ROWS = 100

# "instruction_dataset" for supervised fine-tuning, or "preference_pairs" for DPO.
TRAINING_TYPE = "instruction_dataset"

## Step 1 - Import a dataset

[`datasets.create`](https://docs.adaptionlabs.ai/adaptive-data/create-a-dataset/) pulls the file(s) from Hugging Face and returns a `dataset_id`. You can also `client.datasets.upload_file("local.csv")` or import from Kaggle.

In [4]:
dataset = client.datasets.create(
    source={"url": DATASET_URL, "files": DATASET_FILES},
)
dataset_id = dataset.dataset_id
print("dataset_id:", dataset_id)

dataset_id: ab6559f5-f602-449e-afee-ac924795c0a9


Imports run asynchronously. Poll the status until the row count is populated before adapting.

In [5]:
import time

while True:
    status = client.datasets.get_status(dataset_id)
    if status.status == "failed":
        err = status.error_data
        msg = (err and err.message) or "unknown error"
        raise RuntimeError(f"Ingestion failed: {msg}")
    if status.row_count is not None:
        print(f"Imported {status.row_count} rows")
        break
    time.sleep(5)

Imported 135 rows


## Step 2 - Choose recipes

**Recipes** transform the mapped data. **Brand controls** are your blueprint: `length` sets verbosity and `blueprint` is a freeform instruction that every generated answer is shaped against.

In [6]:
recipe_specification = {
    "recipes": {
        "deduplication": True,
    },
}

brand_controls = {
    "length": "concise",
    "blueprint": "Ground every answer in what is visible in the image.",
}

job_specification = {"max_rows": MAX_ROWS} if MAX_ROWS else {}

## Step 3 - Estimate the run

`estimate=True` validates the configuration and returns a credit quote **without** starting a run.

In [7]:
adapt_kwargs = dict(
    column_mapping=COLUMN_MAPPING,
    training_type=TRAINING_TYPE,
    recipe_specification=recipe_specification,
    brand_controls=brand_controls,
    job_specification=job_specification,
)

estimate = client.datasets.adapt(dataset_id, **adapt_kwargs, estimate=True)
print(f"Estimated credits: {estimate.estimated_credits_consumed}")

Estimated credits: 10.0


## Step 4 - Run the adaptation

In [ ]:
run = client.datasets.adapt(dataset_id, **adapt_kwargs)
print(f"Run started: {run.run_id}")

status = client.datasets.wait_for_completion(dataset_id, timeout=3600)
if status.status == "failed":
    raise RuntimeError(f"Adaptive Data failed: {status.error_data}")
print("Adaptation status:", status.status)

Run started: dataset-ab6559f5-f602-449e-afee-ac924795c0a9-1794951441572


## Step 5 - Check quality

Evaluation can finish after adaptation, so poll with a deadline. Scores are on a 0–10 scale.

In [ ]:
deadline = time.monotonic() + 600
while True:
    evaluation = client.datasets.get_evaluation(dataset_id)
    print("Evaluation status:", evaluation.status)
    if evaluation.status in ("succeeded", "failed", "skipped"):
        break
    if time.monotonic() >= deadline:
        raise TimeoutError("Evaluation did not finish within 10 minutes")
    time.sleep(5)

if evaluation.status == "succeeded" and evaluation.quality:
    q = evaluation.quality
    print(f"Score before: {q.score_before}")
    print(f"Score after:  {q.score_after}")
    print(f"Improvement:  {q.improvement_percent}%")

## Step 6 - Download the adapted dataset

`file_format="parquet"` returns a **gzipped tar of Parquet shards** (`.tar.gz`) - extract it before reading.

In [ ]:
import tarfile

client.datasets.download(dataset_id, file_format="parquet").write_to_file(
    "adapted-dataset.parquet"
)
with tarfile.open("adapted-dataset.tar.gz") as archive:
    archive.extractall("adapted-dataset/")

## Next steps

Vision is the first modality beyond text and documents, not the last. Adaptive Data for vision is available now for teams shipping production image systems.

- [Configure Adaptive Data](https://docs.adaptionlabs.ai/adaptive-data/configure-adaptive-data/) for the full set of recipes and brand controls.
- [Train a model with AutoScientist](https://docs.adaptionlabs.ai/autoscientist-quickstart/) on this adapted dataset.